# Week 5 Deliverable: Bioinformatics Pipeline

## Overview
This notebook implements a complete bioinformatics pipeline for variant calling in CYP genes.

### Genes of Interest
- CYP2C8: chr10:95,036,772-95,069,497
- CYP2C9: chr10:94,938,658-94,990,091
- CYP2C19: chr10:94,762,681-94,855,547

All three genes are located on chromosome 10.


## Step 0: Download Sequencing Data

Download Illumina short-read and PacBio long-read samples.


In [ ]:
%%bash
# Create directories
mkdir -p data results

# Download Illumina short-read data (interleaved paired-end FASTQ)
if [ ! -f data/illumina.fq ]; then
    echo "Downloading Illumina data..."
    wget -qO- https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/illumina.fq.bz2 | bunzip2 > data/illumina.fq
    echo "Illumina data download complete."
else
    echo "data/illumina.fq already exists."
fi

# Download PacBio long-read data
if [ ! -f data/pacbio.fq ]; then
    echo "Downloading PacBio data..."
    wget -qO- https://github.com/inumanag/fall25-csc-bioinf/raw/refs/heads/main/week4/data/pacbio.fq.bz2 | bunzip2 > data/pacbio.fq
    echo "PacBio data download complete."
else
    echo "data/pacbio.fq already exists."
fi

# Check downloaded files
echo ""
echo "Data files:"
ls -lh data/*.fq 2>/dev/null || echo "No FASTQ files found"


## Step 1: Download Reference Genome

Download chromosome 10 from hg38 (GRCh38) as reference.


In [ ]:
%%bash
# Download chr10 reference genome
if [ ! -f results/chr10.fa ]; then
    echo "Downloading chr10 reference genome..."
    wget -q -O results/chr10.fa.gz http://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr10.fa.gz
    gunzip results/chr10.fa.gz
    echo "Download complete."
else
    echo "results/chr10.fa already exists."
fi

# Check file size
ls -lh results/chr10.fa


## Step 2: Alignment with minimap2

Align both samples to the reference genome using appropriate parameters for each technology.


In [ ]:
%%bash
# Align Illumina short reads
echo "Aligning Illumina reads..."
minimap2 -ax sr results/chr10.fa data/illumina.fq | samtools view -bS - | samtools sort -o results/illumina.bam
samtools index results/illumina.bam
echo "Illumina alignment complete."

# Align PacBio long reads
echo "Aligning PacBio reads..."
minimap2 -ax map-pb results/chr10.fa data/pacbio.fq | samtools view -bS - | samtools sort -o results/pacbio.bam
samtools index results/pacbio.bam
echo "PacBio alignment complete."

# Check alignment statistics
echo ""
echo "=== Illumina BAM stats ==="
samtools flagstat results/illumina.bam

echo ""
echo "=== PacBio BAM stats ==="
samtools flagstat results/pacbio.bam


## Step 3: Variant Calling

Call variants in the CYP gene regions using bcftools.


In [ ]:
%%bash
# Define regions of interest (CYP genes)
# Note: BAM file uses "chr10" (with chr prefix)
REGIONS="chr10:94761900-94853205,chr10:94938658-94990091,chr10:95036772-95069497"

# Call variants for Illumina (output uncompressed VCF for HapCUT2)
echo "Calling variants for Illumina..."
bcftools mpileup -f results/chr10.fa -r $REGIONS results/illumina.bam | \
    bcftools call -mv -Ov -o results/illumina.vcf

# Call variants for PacBio (output uncompressed VCF for HapCUT2)
echo "Calling variants for PacBio..."
bcftools mpileup -f results/chr10.fa -r $REGIONS results/pacbio.bam | \
    bcftools call -mv -Ov -o results/pacbio.vcf

echo ""
echo "=== Variant counts ==="
echo "Illumina variants:"
bcftools view -H results/illumina.vcf | wc -l
echo "PacBio variants:"
bcftools view -H results/pacbio.vcf | wc -l


## Step 4: Phasing

Phase variants using HapCUT2 or HapTree-X.


In [ ]:
%%bash
# Phase Illumina variants
echo "Phasing Illumina variants with HapCUT2..."
echo "VCF variants count: $(bcftools view -H results/illumina.vcf | wc -l)"

# Extract haplotype-informative reads
echo "Running extractHAIRS..."
extractHAIRS --bam results/illumina.bam \
    --VCF results/illumina.vcf \
    --out results/illumina_fragments.txt || echo "extractHAIRS failed with exit code $?"

echo "Fragments extracted: $(wc -l < results/illumina_fragments.txt 2>/dev/null || echo 0)"

# Run HapCUT2 for phasing  
echo "Running HAPCUT2..."
HAPCUT2 --fragments results/illumina_fragments.txt \
    --VCF results/illumina.vcf \
    --output results/illumina_phased.hapcut || echo "HAPCUT2 failed with exit code $?"

echo "HapCUT2 output lines: $(wc -l < results/illumina_phased.hapcut 2>/dev/null || echo 0)"

# HapCUT2 automatically outputs phased VCF, rename and compress it
if [ -s results/illumina_phased.hapcut ]; then
    echo "Processing HapCUT2 phased VCF output..."
    mv results/illumina_phased.hapcut.phased.VCF results/illumina_phased.vcf
    
    if [ -s results/illumina_phased.vcf ]; then
        echo "Phased VCF size: $(wc -l < results/illumina_phased.vcf) lines"
        # Compress and index for downstream analysis
        bgzip -f results/illumina_phased.vcf
        bcftools index results/illumina_phased.vcf.gz
        echo "Illumina phasing complete."
    else
        echo "ERROR: Phased VCF file not found"
    fi
else
    echo "WARNING: HapCUT2 produced no output"
    touch results/illumina_phased.vcf
    bgzip -f results/illumina_phased.vcf
fi

# Phase PacBio variants
echo "Phasing PacBio variants with HapCUT2..."

# Extract haplotype-informative reads (PacBio requires --ref for realignment)
extractHAIRS --pacbio 1 \
    --bam results/pacbio.bam \
    --VCF results/pacbio.vcf \
    --ref results/chr10.fa \
    --out results/pacbio_fragments.txt

# Run HapCUT2 for phasing
HAPCUT2 --fragments results/pacbio_fragments.txt \
    --VCF results/pacbio.vcf \
    --output results/pacbio_phased.hapcut

# HapCUT2 automatically outputs phased VCF, rename and compress it
echo "Processing HapCUT2 phased VCF output..."
mv results/pacbio_phased.hapcut.phased.VCF results/pacbio_phased.vcf

if [ -s results/pacbio_phased.vcf ]; then
    echo "Phased VCF size: $(wc -l < results/pacbio_phased.vcf) lines"
    # Compress and index for downstream analysis
    bgzip -f results/pacbio_phased.vcf
    bcftools index results/pacbio_phased.vcf.gz
    echo "PacBio phasing complete."
else
    echo "ERROR: Phased VCF file not found"
    touch results/pacbio_phased.vcf
    bgzip -f results/pacbio_phased.vcf
fi

# Show phasing statistics
echo ""
echo "=== Phasing results ==="
echo "Illumina phased blocks:"
grep "BLOCK" results/illumina_phased.hapcut | wc -l
echo "PacBio phased blocks:"
grep "BLOCK" results/pacbio_phased.hapcut | wc -l

echo ""
echo "Check phased VCF files:"
ls -lh results/*_phased.vcf.gz


## Step 5: Variant Comparison

Compare phased variants between Illumina and PacBio sequencing technologies.


In [ ]:
%%bash
# Compare phased variants using bcftools isec
echo "Comparing Illumina and PacBio phased variants..."

# Create output directory
mkdir -p results/vcf_compare

# Run bcftools isec to find shared and unique variants
# sites.txt contains presence information: "11"=shared, "10"=illumina-only, "01"=pacbio-only
bcftools isec illumina_phased.vcf.gz pacbio_phased.vcf.gz -p results/vcf_compare 2>/dev/null

# Parse sites.txt
shared=$(awk '$5=="11"' results/vcf_compare/sites.txt | wc -l)
illumina_only=$(awk '$5=="10"' results/vcf_compare/sites.txt | wc -l)
pacbio_only=$(awk '$5=="01"' results/vcf_compare/sites.txt | wc -l)
illumina_total=$((illumina_only + shared))
pacbio_total=$((pacbio_only + shared))

echo ""
echo "=== Overall Concordance Statistics ==="
echo "Total Illumina phased variants: $illumina_total"
echo "Total PacBio phased variants: $pacbio_total"
echo "Shared variants: $shared"
echo "Illumina-only variants: $illumina_only"
echo "PacBio-only variants: $pacbio_only"

echo ""
echo "Concordance:"
concordance_illumina=$(awk "BEGIN {printf \"%.1f\", $shared/$illumina_total*100}")
concordance_pacbio=$(awk "BEGIN {printf \"%.1f\", $shared/$pacbio_total*100}")
echo "  - ${concordance_illumina}% of Illumina variants are shared"
echo "  - ${concordance_pacbio}% of PacBio variants are shared"

echo ""
echo "========================================"
echo "=== PER-GENE ANALYSIS ==="
echo "========================================"

# Define gene regions
# CYP2C19: chr10:94761900-94853205
# CYP2C9:  chr10:94938658-94990091  
# CYP2C8:  chr10:95036772-95069497

echo ""
echo "--- CYP2C19 (chr10:94761900-94853205) ---"
cyp2c19_illumina_only=$(awk '$1=="chr10" && $2>=94761900 && $2<=94853205 && $5=="10"' results/vcf_compare/sites.txt | wc -l)
cyp2c19_pacbio_only=$(awk '$1=="chr10" && $2>=94761900 && $2<=94853205 && $5=="01"' results/vcf_compare/sites.txt | wc -l)
cyp2c19_shared=$(awk '$1=="chr10" && $2>=94761900 && $2<=94853205 && $5=="11"' results/vcf_compare/sites.txt | wc -l)
cyp2c19_illumina_total=$((cyp2c19_illumina_only + cyp2c19_shared))
cyp2c19_pacbio_total=$((cyp2c19_pacbio_only + cyp2c19_shared))

echo "  Total Illumina variants: $cyp2c19_illumina_total"
echo "  Total PacBio variants: $cyp2c19_pacbio_total"
echo "  Shared variants: $cyp2c19_shared"
echo "  Illumina-only: $cyp2c19_illumina_only"
echo "  PacBio-only: $cyp2c19_pacbio_only"
if [ $cyp2c19_illumina_total -gt 0 ]; then
    concordance=$(awk "BEGIN {printf \"%.1f\", $cyp2c19_shared/$cyp2c19_illumina_total*100}")
    echo "  Concordance: ${concordance}%"
fi

echo ""
echo "  Top Illumina-only variants in CYP2C19:"
awk '$1=="chr10" && $2>=94761900 && $2<=94853205 && $5=="10" {print "    "$1":"$2" "$3">"$4}' results/vcf_compare/sites.txt | head -3

echo "  Top PacBio-only variants in CYP2C19:"
awk '$1=="chr10" && $2>=94761900 && $2<=94853205 && $5=="01" {print "    "$1":"$2" "$3">"$4}' results/vcf_compare/sites.txt | head -3

echo ""
echo "--- CYP2C9 (chr10:94938658-94990091) ---"
cyp2c9_illumina_only=$(awk '$1=="chr10" && $2>=94938658 && $2<=94990091 && $5=="10"' results/vcf_compare/sites.txt | wc -l)
cyp2c9_pacbio_only=$(awk '$1=="chr10" && $2>=94938658 && $2<=94990091 && $5=="01"' results/vcf_compare/sites.txt | wc -l)
cyp2c9_shared=$(awk '$1=="chr10" && $2>=94938658 && $2<=94990091 && $5=="11"' results/vcf_compare/sites.txt | wc -l)
cyp2c9_illumina_total=$((cyp2c9_illumina_only + cyp2c9_shared))
cyp2c9_pacbio_total=$((cyp2c9_pacbio_only + cyp2c9_shared))

echo "  Total Illumina variants: $cyp2c9_illumina_total"
echo "  Total PacBio variants: $cyp2c9_pacbio_total"
echo "  Shared variants: $cyp2c9_shared"
echo "  Illumina-only: $cyp2c9_illumina_only"
echo "  PacBio-only: $cyp2c9_pacbio_only"
if [ $cyp2c9_illumina_total -gt 0 ]; then
    concordance=$(awk "BEGIN {printf \"%.1f\", $cyp2c9_shared/$cyp2c9_illumina_total*100}")
    echo "  Concordance: ${concordance}%"
fi

echo ""
echo "  Top Illumina-only variants in CYP2C9:"
awk '$1=="chr10" && $2>=94938658 && $2<=94990091 && $5=="10" {print "    "$1":"$2" "$3">"$4}' results/vcf_compare/sites.txt | head -3

echo "  Top PacBio-only variants in CYP2C9:"
awk '$1=="chr10" && $2>=94938658 && $2<=94990091 && $5=="01" {print "    "$1":"$2" "$3">"$4}' results/vcf_compare/sites.txt | head -3

echo ""
echo "--- CYP2C8 (chr10:95036772-95069497) ---"
cyp2c8_illumina_only=$(awk '$1=="chr10" && $2>=95036772 && $2<=95069497 && $5=="10"' results/vcf_compare/sites.txt | wc -l)
cyp2c8_pacbio_only=$(awk '$1=="chr10" && $2>=95036772 && $2<=95069497 && $5=="01"' results/vcf_compare/sites.txt | wc -l)
cyp2c8_shared=$(awk '$1=="chr10" && $2>=95036772 && $2<=95069497 && $5=="11"' results/vcf_compare/sites.txt | wc -l)
cyp2c8_illumina_total=$((cyp2c8_illumina_only + cyp2c8_shared))
cyp2c8_pacbio_total=$((cyp2c8_pacbio_only + cyp2c8_shared))

echo "  Total Illumina variants: $cyp2c8_illumina_total"
echo "  Total PacBio variants: $cyp2c8_pacbio_total"
echo "  Shared variants: $cyp2c8_shared"
echo "  Illumina-only: $cyp2c8_illumina_only"
echo "  PacBio-only: $cyp2c8_pacbio_only"
if [ $cyp2c8_illumina_total -gt 0 ]; then
    concordance=$(awk "BEGIN {printf \"%.1f\", $cyp2c8_shared/$cyp2c8_illumina_total*100}")
    echo "  Concordance: ${concordance}%"
fi

echo ""
echo "  Top Illumina-only variants in CYP2C8:"
awk '$1=="chr10" && $2>=95036772 && $2<=95069497 && $5=="10" {print "    "$1":"$2" "$3">"$4}' results/vcf_compare/sites.txt | head -3

echo "  Top PacBio-only variants in CYP2C8:"
awk '$1=="chr10" && $2>=95036772 && $2<=95069497 && $5=="01" {print "    "$1":"$2" "$3">"$4}' results/vcf_compare/sites.txt | head -3


## Step 5.5: Generate IGV Screenshots

Automatically generate IGV screenshots for selected variants.


In [ ]:
%%bash
cd week5

echo "=== Generating IGV Batch Script for Auto Screenshots ==="

# Create output directory for auto-generated IGV images
mkdir -p igv_images/auto

# Create IGV batch script
cat > igv_batch_script.txt << 'EOF'
# IGV Batch Script for Automated Screenshots
new
genome hg38
snapshotDirectory igv_images/auto
preference SAM.SHOW_SOFT_CLIPPED true

# Load alignment files
load illumina_sorted.bam
load pacbio_sorted.bam

# Screenshot 1: Illumina-only variant in CYP2C19
goto chr10:94772788
collapse
maxPanelHeight 500
snapshot auto_chr10_94772788_illumina_only.png

# Screenshot 2: PacBio-only variant in CYP2C9
goto chr10:94947469
collapse
maxPanelHeight 500
snapshot auto_chr10_94947469_pacbio_only.png

# Screenshot 3: Shared variant at CYP2C19 start
goto chr10:94761900
collapse
maxPanelHeight 500
snapshot auto_chr10_94761900_shared.png

exit
EOF

echo "✓ IGV batch script created: igv_batch_script.txt"
echo ""
echo "To generate automated screenshots:"
echo "  1. Open IGV (Integrative Genomics Viewer)"
echo "  2. Tools > Run Batch Script..."
echo "  3. Select: week5/igv_batch_script.txt"
echo ""
echo "Auto-generated screenshots will be saved to: week5/igv_images/auto/"
echo "(This will NOT overwrite existing manual screenshots)"
echo ""
echo "Variants to inspect:"
echo "  1. chr10:94772788 - Illumina-only (CYP2C19)"
echo "  2. chr10:94947469 - PacBio-only (CYP2C9)"
echo "  3. chr10:94761900 - Shared variant (CYP2C19)"


### Screenshots and Discussion: Are discordant variants true variants or sequencing artifacts?

![Illumina-only variant](./igv_images/illumina_only.png)

### IGV Screenshot: chr10:94,792,532–94,792,572

- **Observation:**  
  In Illumina reads (top track), a consistent G→T substitution is observed near position 94,792,554.  
  Multiple overlapping short reads support this change (red mismatch markers).  
  In contrast, PacBio reads (bottom track) show no evidence of the substitution.

- **Interpretation:**  
  The variant appears only in Illumina sequencing.  
  Given PacBio’s low coverage (1x) at this locus, the absence of evidence may not confirm or refute the variant.  
  Alternatively, this could represent an Illumina alignment artifact in a repetitive or GC-rich region.

- **Conclusion:**  
  This is an *Illumina-only variant*.  
  Further phasing or deeper long-read coverage is required to determine whether this SNP is genuine.

![Pacbio-only variant](./igv_images/pacbio_only.png)
### PacBio-only variant at chr10:94947469 (CYP2C9)

- **Observation:**  
  PacBio reads show a small subset (~10–20%) supporting a C substitution, while the majority remain consistent with the reference base (T). Illumina reads show no evidence of this variant.

- **Interpretation:**  
  The low variant allele fraction (VAF) suggests that this may represent a sequencing artifact or low-confidence site rather than a true SNP. Additional filtering or replicate data would be required to confirm.

- **Conclusion:**  
  Classified as a low-confidence PacBio-only variant.
  Unlike the high-confidence PacBio-only deletion observed elsewhere, this site likely reflects stochastic PacBio substitution noise rather than a biological variant.


![Shared variant](./igv_images/shared.png)
### Shared variant at chr10:94761900 (CYP2C19)

- **Observation:**  
  Both Illumina (top) and PacBio (bottom) reads show a clear SNP at the same position.  
  The color pattern (red/blue) and allele frequency are consistent across platforms.

- **Interpretation:**  
  This represents a *true positive* shared variant confirmed by two independent sequencing technologies.  
  High agreement indicates high confidence in variant calling and alignment accuracy.

- **Conclusion:**  
  This locus can be classified as a **shared high-confidence SNP**, validating the variant detection consistency between Illumina and PacBio pipelines.


## Step 6: Star-Allele Identification

Identify star-alleles using PharmVar database.


In [ ]:
# TODO: Implement star-allele identification
print("Star-allele identification to be implemented")


## Time Estimate

Estimated time to complete this assignment: 8-12 hours

Breakdown:
- Understanding requirements: 1 hour
- Setting up tools and environment: 1 hour
- Downloading and aligning data: 2 hours
- Variant calling and phasing: 2-3 hours
- Variant comparison and analysis: 2-3 hours
- Star-allele identification: 1-2 hours
- Documentation and cleanup: 1 hour
